# FAA_CxER — exploration notebook

This is a thin, interactive companion to the CLI scripts in `scripts/`. It
exists for poking at the data and sanity-checking a figure inline — the
actual pipeline stages are meant to be run as scripts (see the top-level
README.md), not copy-pasted into notebook cells like the old FAA_EMNLP
notebooks did.

Run stage 3 (`python scripts/03_merge_cxer_labels.py`) at least once
before this notebook, so `data/predictions/FAA_multi_model_predictions_with_metrics_and_cxer.json`
exists.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from faa_cxer.config import default_config, DEFAULT_MODEL_PAIRS
from faa_cxer.io_utils import load_json, load_records

cfg = default_config()
cfg

## Load the merged dataset (stage 3 output)

In [ ]:
data = load_json(cfg.predictions_with_metrics_and_cxer)
samples = data["samples"]
print(f"{len(samples)} samples, {len(data['corpus_metrics'])} scored ASR models")
data["corpus_metrics"]["FAA_Model"]

## The headline result: WER drops far more than CxER

This is the 2x2 grouped bar chart from `scripts/07_make_comparison_plots.py`,
rendered inline instead of saved straight to a file.

In [ ]:
from faa_cxer.viz import ComparisonBarChart

chart = ComparisonBarChart(
    DEFAULT_MODEL_PAIRS,
    corpus_metrics=data["corpus_metrics"],
    corpus_cxer_metrics=data.get("corpus_cxer_metrics", {}),
)
chart.plot(str(cfg.outputs_dir / "faa_pretrained_vs_finetuned.png"), show=True)

## The paradox, sample by sample

`ParadoxAnalyzer` finds individual utterances with low WER (looks like a
great transcription) that the LLM judge still flagged as a critical,
operationally unsafe error.

In [ ]:
from faa_cxer.analysis import ParadoxAnalyzer

analyzer = ParadoxAnalyzer(samples, judge=cfg.judge_short_name())
pair = DEFAULT_MODEL_PAIRS[0]  # Whisper Medium
rows = analyzer.get_model_records(pair.finetuned_key)
low_wer_critical = [
    r for r in rows
    if r["classification"] == "Critical_Errors" and r["WER"] is not None and r["WER"] < 0.1
]

print(f"{len(low_wer_critical)} samples with WER < 0.1 but a critical error")
for r in low_wer_critical[:5]:
    print()
    print(f"  WER={r['WER']:.3f}")
    print(f"  ref: {r['reference']}")
    print(f"  hyp: {r['hypothesis']}")

## Next steps

- `scripts/04_analyze_token_frequency.py` — does fine-tuning help common
  words more than rare ones?
- `scripts/05_analyze_transitions.py` — the safety-drift transition matrix
  and Sankey diagram.
- `scripts/06_analyze_paradox.py` — the full concentration / trust-violation
  table and per-family scatter plots.

See the top-level README.md for the full pipeline explanation.